# Day 16 — Window Functions I: ROW_NUMBER, RANK, DENSE_RANK
**Estimated time:** 60–75 min
**Datasets:** `sales_orders.csv`, `bw_sales_kpis.csv`

## Learning Objectives
- Understand the anatomy of the OVER() clause: PARTITION BY, ORDER BY, frame
- Know the difference between ROW_NUMBER, RANK, and DENSE_RANK — cold
- Apply ROW_NUMBER for deduplication and top-N-per-group pattern

In [ ]:
import duckdb
import pandas as pd
import numpy as np
from pathlib import Path

from analytics_bootcamp.config import RAW_DATA_DIR as DATA_DIR
# Load all CSVs
inv   = pd.read_csv(f"{DATA_DIR}/materials_inventory.csv")
sales = pd.read_csv(f"{DATA_DIR}/sales_orders.csv")
cc    = pd.read_csv(f"{DATA_DIR}/cost_center_actuals.csv")
hr    = pd.read_csv(f"{DATA_DIR}/hr_headcount.csv")
bw    = pd.read_csv(f"{DATA_DIR}/bw_sales_kpis.csv")

# Normalize column names
for df in [inv, sales, cc, hr, bw]:
    df.columns = [c.strip().upper() for c in df.columns]

# In-memory DuckDB — register pandas DataFrames as tables
con = duckdb.connect()
con.register("MATERIALS",   inv)
con.register("SALES",       sales)
con.register("COST_CENTER", cc)
con.register("HR",          hr)
con.register("BW_SALES",    bw)

def q(sql):
    return con.sql(sql).df()

# Verify
for tbl, df in [("MATERIALS",inv),("SALES",sales),("COST_CENTER",cc),("HR",hr),("BW_SALES",bw)]:
    n = q(f"SELECT COUNT(*) AS n FROM {tbl}").iloc[0,0]
    print(f"  {tbl:15s}: {n:,} rows")


In [ ]:
# -- OVER() clause anatomy --
# OVER() = window definition
# PARTITION BY = restart numbering for each group
# ORDER BY     = defines ranking/sequencing within the partition
# ROWS/RANGE BETWEEN = frame (how many rows around the current row)
#
# Comparing all three ranking functions side by side:
result = q(
    """
    SELECT
        VKORG,
        KUNNR,
        SUM(NETWR) AS revenue,
        ROW_NUMBER() OVER (PARTITION BY VKORG ORDER BY SUM(NETWR) DESC) AS row_num,
        RANK()       OVER (PARTITION BY VKORG ORDER BY SUM(NETWR) DESC) AS rnk,
        DENSE_RANK() OVER (PARTITION BY VKORG ORDER BY SUM(NETWR) DESC) AS dense_rnk
    FROM SALES
    GROUP BY VKORG, KUNNR
    ORDER BY VKORG, revenue DESC
    """
)
display(result.head(30))
print("\nKey difference: when two customers have identical revenue,")
print("  RANK skips the next number, DENSE_RANK does not.")

In [ ]:
# -- ROW_NUMBER vs RANK vs DENSE_RANK: illustrate with tie --
# Construct a minimal example with forced ties
result = q(
    """
    WITH scores(name, score) AS (
        VALUES ('Alice', 100), ('Bob', 100), ('Carol', 90), ('Dave', 85)
    )
    SELECT
        name,
        score,
        ROW_NUMBER() OVER (ORDER BY score DESC) AS row_number,
        RANK()       OVER (ORDER BY score DESC) AS rank,
        DENSE_RANK() OVER (ORDER BY score DESC) AS dense_rank
    FROM scores
    """
)
display(result)
# Alice and Bob both score 100:
# ROW_NUMBER: 1, 2 (arbitrary tiebreak — order not guaranteed)
# RANK:       1, 1 -- then 3 (gap)
# DENSE_RANK: 1, 1 -- then 2 (no gap)

In [ ]:
# -- Deduplication with ROW_NUMBER: keep latest order per customer --
result = q(
    """
    WITH ranked_orders AS (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY KUNNR ORDER BY ERDAT DESC) AS rn
        FROM SALES
    )
    SELECT VBELN, KUNNR, ERDAT, NETWR, STATUS
    FROM ranked_orders
    WHERE rn = 1
    ORDER BY NETWR DESC
    LIMIT 15
    """
)
display(result)

In [ ]:
# -- Top-N per group: top 3 customers by revenue per sales org --
result = q(
    """
    WITH customer_revenue AS (
        SELECT VKORG, KUNNR, SUM(NETWR) AS total_revenue
        FROM SALES
        GROUP BY VKORG, KUNNR
    ),
    ranked AS (
        SELECT
            VKORG,
            KUNNR,
            total_revenue,
            DENSE_RANK() OVER (PARTITION BY VKORG ORDER BY total_revenue DESC) AS revenue_rank
        FROM customer_revenue
    )
    SELECT VKORG, KUNNR, total_revenue, revenue_rank
    FROM ranked
    WHERE revenue_rank <= 3
    ORDER BY VKORG, revenue_rank
    """
)
display(result)

In [ ]:
# -- NTILE: quartile bucketing of materials by inventory value --
result = q(
    """
    SELECT
        MATNR,
        MAKTX,
        LABST * STPRS AS inv_value,
        NTILE(4) OVER (ORDER BY LABST * STPRS DESC) AS quartile
    FROM MATERIALS
    WHERE LABST > 0
    ORDER BY inv_value DESC
    LIMIT 20
    """
)
display(result)

In [ ]:
# -- Percent rank and cumulative distribution --
result = q(
    """
    WITH material_values AS (
        SELECT MATNR, MAKTX, LABST * STPRS AS inv_value
        FROM MATERIALS WHERE LABST > 0
    )
    SELECT
        MATNR,
        MAKTX,
        ROUND(inv_value, 2) AS inv_value,
        ROUND(PERCENT_RANK() OVER (ORDER BY inv_value), 4) AS pct_rank,
        ROUND(CUME_DIST()   OVER (ORDER BY inv_value), 4) AS cume_dist
    FROM material_values
    ORDER BY inv_value DESC
    LIMIT 15
    """
)
display(result)

---
## Daily Prompt

**Rank each material by revenue within its material type. Return only the top 3 per material type. Show: rank, material, material type, and revenue.**

```sql
-- YOUR SOLUTION
WITH material_revenue AS (
    SELECT
        s.MATNR,
        m.MATERIAL_TYPE,
        SUM(s.NETWR) AS total_revenue
    FROM SALES s
    INNER JOIN MATERIALS m ON s.MATNR = m.MATNR
    GROUP BY s.MATNR, m.MATERIAL_TYPE
),
ranked AS (
    SELECT
        MATNR,
        MATERIAL_TYPE,
        total_revenue,
        -- YOUR SOLUTION: DENSE_RANK() OVER (PARTITION BY ... ORDER BY ...)
    FROM material_revenue
)
SELECT *
FROM ranked
WHERE -- YOUR SOLUTION: filter top 3
ORDER BY MATERIAL_TYPE, revenue_rank
```

> **Hint:** Use `DENSE_RANK()` rather than `RANK()` so ties do not skip ranks, and you get exactly 3 materials per type (or more in case of exact revenue ties).